# Notebook 15 — Pretraining a Tiny Decoder with Native PyTorch

    ## Learning objectives

    - Turn raw text into causal language-model examples without a framework abstraction
- Build and train a randomly initialized decoder for one small epoch
- Evaluate loss, perplexity, generation, and checkpoint round trips honestly

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if IN_COLAB and not token:
    from google.colab import userdata
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            token = None
        if token:
            break
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub variable. The course also sets its
# descriptive alias because some lesson code uses HUGGINGFACE_TOKEN explicitly.
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGINGFACE_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if True and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("Add an HF_TOKEN secret in Colab, enable notebook access, then rerun this cell.")


## 15.1 What this experiment proves—and what it does not

Pretraining begins with random parameters and optimizes next-token likelihood over a large,
broad corpus. This notebook preserves that causal chain at toy scale: raw text becomes bytes,
bytes become fixed-length examples, a decoder predicts the following byte, and AdamW changes
every parameter. One epoch is enough to verify mechanics and watch loss fall, but not enough
to create a generally useful language model. Real runs differ by many orders of magnitude in
data, parameters, compute, validation breadth, and operational controls.

We use bytes so encoding is lossless and needs no learned tokenizer. IDs 0–255 represent byte
values and ID 256 marks document boundaries. The price is longer sequences and predictions
that operate below human-visible characters. This makes a byte tokenizer excellent teaching
machinery, not an automatic production choice.


In [ ]:
import math, random, torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42); random.seed(42)
device = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
EOS, VOCAB_SIZE, SEQ_LEN = 256, 257, 64
documents = [
    "A language model estimates the next token from the tokens before it.",
    "Attention mixes information across earlier positions in a causal sequence.",
    "Training data quality, coverage, and provenance shape model behavior.",
    "Validation loss estimates generalization to held-out text.",
    "Small experiments verify code; they do not establish broad capability.",
    "Gradient descent updates parameters to reduce average negative log likelihood.",
]
train_docs, valid_docs = documents[:5], documents[5:]
def encode(text): return list(text.encode("utf-8"))
def decode(ids): return bytes(i for i in ids if i < 256).decode("utf-8", errors="replace")
print("device:", device, "example IDs:", encode("LLM") + [EOS])


In [ ]:
class ByteBlocks(Dataset):
    def __init__(self, docs, repeats, sequence_length):
        stream = []
        for _ in range(repeats):
            for doc in docs: stream.extend(encode(doc) + [EOS])
        self.tokens = torch.tensor(stream, dtype=torch.long)
        self.starts = list(range(0, len(stream) - sequence_length - 1, sequence_length))
        self.sequence_length = sequence_length
    def __len__(self): return len(self.starts)
    def __getitem__(self, index):
        start = self.starts[index]
        window = self.tokens[start:start + self.sequence_length + 1]
        return window[:-1], window[1:]

train_data = ByteBlocks(train_docs, repeats=30, sequence_length=SEQ_LEN)
valid_data = ByteBlocks(valid_docs, repeats=8, sequence_length=SEQ_LEN)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=16)
x, y = next(iter(train_loader))
print("examples:", len(train_data), "batch:", x.shape, "shift correct:", torch.equal(x[:, 1:], y[:, :-1]))


## 15.2 A small modern decoder

Each block is pre-normalized: causal self-attention and a feed-forward network each add a
residual update. PyTorch's scaled-dot-product attention can select an optimized kernel on
supported hardware. Learned position embeddings keep this implementation compact; Notebook 8
develops RoPE, and Notebook 9 explains why fused attention changes memory traffic rather than
the mathematical attention result. Input and output embeddings are tied, reducing parameters
and forcing both interfaces to share a token geometry.


In [ ]:
class Block(nn.Module):
    def __init__(self, width, heads):
        super().__init__(); self.heads = heads; self.head_dim = width // heads
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.qkv, self.proj = nn.Linear(width, 3 * width), nn.Linear(width, width)
        self.ff = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
    def forward(self, x):
        b, t, c = x.shape
        q, k, v = self.qkv(self.norm1(x)).chunk(3, dim=-1)
        shape = (b, t, self.heads, self.head_dim)
        q, k, v = [z.view(shape).transpose(1, 2) for z in (q, k, v)]
        attended = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(attended.transpose(1, 2).contiguous().view(b, t, c))
        return x + self.ff(self.norm2(x))

class TinyDecoder(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, width=96, layers=3, heads=4, max_length=SEQ_LEN):
        super().__init__(); self.max_length = max_length
        self.token = nn.Embedding(vocab, width); self.position = nn.Embedding(max_length, width)
        self.blocks = nn.ModuleList([Block(width, heads) for _ in range(layers)])
        self.norm = nn.LayerNorm(width); self.head = nn.Linear(width, vocab, bias=False)
        self.head.weight = self.token.weight
    def forward(self, ids, labels=None):
        positions = torch.arange(ids.shape[1], device=ids.device)
        hidden = self.token(ids) + self.position(positions)
        for block in self.blocks: hidden = block(hidden)
        logits = self.head(self.norm(hidden))
        loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten()) if labels is not None else None
        return logits, loss

model = TinyDecoder().to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
@torch.no_grad()
def generate(model, prompt, new_tokens=80, temperature=0.8):
    ids = torch.tensor([encode(prompt)], device=device)
    for _ in range(new_tokens):
        context = ids[:, -model.max_length:]
        logits, _ = model(context)
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        nxt = torch.multinomial(probs, 1)
        ids = torch.cat((ids, nxt), dim=1)
    return decode(ids[0].tolist())

print("BEFORE:", repr(generate(model, "Training ", 50)))


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.1)
model.train(); running = []
for inputs, labels in train_loader:                 # exactly one epoch
    inputs, labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(inputs, labels)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step(); running.append(loss.item())
print(f"one-epoch mean train loss: {sum(running)/len(running):.3f}")


In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval(); weighted_loss = 0.0; tokens = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        _, loss = model(inputs, labels)
        weighted_loss += loss.item() * labels.numel(); tokens += labels.numel()
    mean = weighted_loss / tokens
    return {"loss": mean, "perplexity": math.exp(min(mean, 20)), "tokens": tokens}

print("validation:", evaluate(valid_loader))
print("AFTER:", repr(generate(model, "Training ", 80)))


In [ ]:
from pathlib import Path
checkpoint = Path("artifacts/native_pretraining/tiny_decoder.pt")
checkpoint.parent.mkdir(parents=True, exist_ok=True)
torch.save({"model": model.state_dict(), "config": {"vocab": VOCAB_SIZE, "max_length": SEQ_LEN}}, checkpoint)
restored = TinyDecoder().to(device)
restored.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True)["model"])
restored.eval()
probe = torch.tensor([encode("A model")], device=device)
with torch.no_grad():
    print("checkpoint exact:", torch.equal(model(probe)[0], restored(probe)[0]))


## 15.3 Reading a pretraining run like an experiment

The unit of progress is tokens, not epochs. An epoch over a duplicated toy corpus is merely a
convenient bounded loop; large pretraining corpora may be traversed once or not even have a
meaningful epoch boundary. Record unique documents, raw and post-filter tokens, repeated tokens,
sequence length, padding fraction, optimizer updates, effective token batch, and total compute.
Two “one epoch” runs can represent radically different amounts of learning.

Cross-entropy averages surprise at the target tokens. Perplexity is its exponential and is only
comparable when tokenizer, tokenization, evaluation text, boundary handling, and masking match.
Byte-level perplexity and GPT-style subword perplexity are therefore not directly comparable.
Generation samples are useful qualitative probes, but a lucky completion cannot replace held-out
loss and capability tests. Before trusting a curve, deliberately overfit one batch, verify the
input/label shift, inspect masks, and compare a checkpoint round trip on fixed logits.

The tiny corpus repeats phrases so a learner can see a result quickly. This creates memorization
and distribution leakage by design. A real split occurs before deduplication-aware packing and is
separated by document, source, author, or time where appropriate. Data cards should capture
provenance, license, languages, filtering, personal-information policy, and known blind spots.


In [ ]:
# Translate loop settings into the quantities a run report should state.
batch_size = train_loader.batch_size
updates = len(train_loader)
tokens_per_update = batch_size * SEQ_LEN
print({"optimizer_updates": updates,
       "nominal_tokens_per_update": tokens_per_update,
       "nominal_epoch_tokens": updates * tokens_per_update,
       "unique_source_documents": len(train_docs)})


## 15.4 Scaling beyond the demonstration

Scaling first stresses data delivery and failure recovery. Shard immutable tokenized data; shuffle
reproducibly across workers; save model, optimizer, scheduler, scaler, RNG, and sampler position;
and test resumption early. Mixed precision reduces memory and raises throughput on supported GPUs.
Gradient accumulation raises effective batch without fitting more activations at once. Activation
checkpointing trades recomputation for memory. DDP replicates the model, while FSDP/ZeRO shard
state; tensor and pipeline parallelism become relevant when layers cannot fit on one accelerator.

Monitor validation loss by domain, gradient norm, learning rate, tokens/second, hardware
utilization, data-loader stalls, numerical overflows, memory, and checkpoint health. Scaling a
silent label bug only makes an expensive bug. Notebooks 10 and 11 develop optimization and
distributed mechanics after this end-to-end anchor.

Initialization and optimization interact with scale. Residual branches, normalization, embedding
variance, learning-rate warmup, and weight decay determine whether signals remain numerically useful
through depth. Seed every relevant generator for debugging, but repeat important conclusions across
seeds because a single tiny run has high variance. Inspect parameter update norms relative to
parameter norms, not only scalar loss. If loss falls implausibly quickly, check duplicated validation
text and boundary leakage. If it remains near the uniform baseline `log(vocabulary_size)`, verify
labels, causal alignment, and optimizer updates before changing the architecture.


## 15.5 Diagnose a pretraining curve

Log token-weighted training loss, held-out loss, learning rate, gradient norm, tokens per second, valid tokens, and elapsed compute on the same update axis. A smooth training curve can hide duplicated data, broken evaluation, or a model learning only local statistics. Compare with a unigram or n-gram baseline and fixed generation probes. Sudden spikes call for preserving the offending batch and checkpoint before changing clipping or learning rate. One epoch demonstrates plumbing; it does not establish useful language capability.


In [ ]:
history=[{"step":0,"train":4.2,"valid":4.25},{"step":100,"train":3.1,"valid":3.3},{"step":200,"train":2.6,"valid":3.35}]
for row in history: print(row,"gap",row["valid"]-row["train"]); print("best valid",min(history,key=lambda x:x["valid"]))


## 15.6 Sampling the trained decoder

Generation repeatedly applies the same causal model used for loss, selects only the final-position logits, applies constraints and a decoding rule, appends a token, and stops on EOS or budget. Temperature rescales logits; top-k and top-p truncate support; greedy decoding is deterministic but not universally more correct. Cache-free generation is appropriate for this tiny reference model because it exposes the algorithm. Always reload the saved artifact before the final demonstration so the lesson proves the checkpoint is usable.


In [ ]:
def sample_logits(logits,temperature=1.0,top_k=None,generator=None):
 scaled=logits/max(temperature,1e-6)
 if top_k: scaled[scaled < torch.topk(scaled,top_k).values[-1]]=float("-inf")
 return torch.multinomial(scaled.softmax(-1),1,generator=generator).item()
g=torch.Generator().manual_seed(2); print([sample_logits(torch.tensor([2.,1.,.5]),.8,2,g) for _ in range(8)])


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [PyTorch Transformer building blocks](https://docs.pytorch.org/tutorials/intermediate/transformer_building_blocks.html)
- [AdamW](https://arxiv.org/abs/1711.05101)


## Exercises

    1. Replace learned positions with the RoPE implementation from Notebook 8.
2. Add validation checkpoints during the epoch and plot train versus validation loss.
3. Increase corpus diversity while holding token count fixed; explain the changed generations.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
